# **US Flights Delay:** Schema Engineering

## Imports

In [1]:
import sys

In [2]:
sys.path.append("../src/database")
sys.path.append("../src/database/queries")

In [3]:
import connection 
import base_queries
import query_executor

## Connect to client and database

In [4]:
MONGODB_URI = "mongodb://localhost:27017/"
MONGODB_NAME = "flights_delay_db"

In [5]:
client, database = connection.connect_to_database(uri = MONGODB_URI, db_name = MONGODB_NAME)

In [6]:
database.list_collection_names()

['airports',
 'airports_summary_hybrid_optimized',
 'runways',
 'cancelled_deverted_2023',
 'flights_hybrid_optimized',
 'us_flights_2023',
 'weather_hybrid_optimized',
 'airport_frequencies',
 'airports_geolocation',
 'us_flights_optimized',
 'weather_meteo_by_airport']

## Queries

### **Flight Delay Analysis:** Base Queries

In [8]:
base_queries = base_queries.get_base_queries()

In [7]:
collection = database["us_flights_2023"]

#### **Q1:** Which US states have the highest average delays by season (Winter, Spring, Summer, Fall)?

Data from the **`us_flights_2023`** and **`airports_geolocation`** collections are analyzed.

The season is determined based on the `flight_date` field:
* **Winter:** December–February
* **Spring:** March–May
* **Summer:** June–August
* **Fall:** September–November

The average delay is calculated as `avg(dep_delay)` for all flights from a given state (`state` from `airports_geolocation`) in a given season.

**Result:** List of US states with average delay by season, sorted in descending order of average delay.

In [8]:
base_query_1 = base_queries['query_1']

In [9]:
results, execution_time = query_executor.execute_query(collection, base_query_1, "Query 1")

Query 'Query 1' executed in 412.3152 seconds


#### **Q2:** Which airlines have the highest average delays on rainy days?

Collections **`us_flights_2023`** and **`weather_meteo_by_airport`** are used.

Precipitation is taken from the `prcp' field (mm).

**Significant precipitation**: days when `prcp > 5.0`.

Need to find average delay (`avg(dep_delay)`) by airline (`airline`) only for days with significant precipitation, based on weather conditions from `departure.airport_code`.

**Result:** Airlines with average delay on days with precipitation > 5 mm, sorted in descending order of value.

In [ ]:
# Index for weather_meteo_by_airport
database["weather_meteo_by_airport"].create_index([
    ("airport_id", 1),
    ("time", 1),
    ("prcp", 1)
])

# Index for us_flights_2023
database["us_flights_2023"].create_index([
    ("Dep_Airport", 1),
    ("FlightDate", 1),
    ("Airline", 1)
])

In [ ]:
base_query_2 = base_queries['query_2']

#### **Q3:** Which airports have the most canceled flights during bad weather?

Collections **`cancelled_deverted_2023`**, **`weather_meteo_by_airport`** and **`airports_geolocation`** are used.

Canceled flights are those with `cancelled = 1`.

**Bad weather conditions** are defined as:
* `prcp > 10 mm' *(heavy precipitation)*
* **or** `wspd > 15 m/s' *(strong wind)*

The query should match ($lookup) flights and weather data by `dep_airport` and `airport_id`.

**Result:** Airports with the highest number of canceled flights during bad weather, sorted in descending order of cancellations.

In [ ]:
# Index for cancelled_deverted_2023
database["cancelled_deverted_2023"].create_index([
    ("Cancelled", 1),
    ("Dep_Airport", 1),
    ("FlightDate", 1)
], name="cancelled_flights_match")
    
# Index for weather_meteo_by_airport  
database["weather_meteo_by_airport"].create_index([
    ("airport_id", 1),
    ("time", 1),
    ("prcp", 1),
    ("wspd", 1)
], name="weather_lookup_optimized")
    
database["weather_meteo_by_airport"].create_index([
    ("airport_id", 1),
    ("time", 1),
    ("wspd", 1) 
], name="weather_wind_lookup")
    
# Index for airports_geolocation
database["airports_geolocation"].create_index([
    ("IATA_CODE", 1)
], name="airport_geo_lookup")

In [ ]:
base_query_3 = base_queries['query_3']

#### **Q4:** Do airports with more diverse runways have lower average delays?

Collections **`airports`**, **`runways`**, and **`us_flights_2023`** are used.

For each airport, the following is calculated:
* **Runway Diversity Index (RDI)** = number of different values ​​of `surface` from the collection of `runways` per airport.
* **Average Delay** = average `dep_delay` from `us_flights_2023` per airport.

It is necessary to merge (`$lookup') all three collections, calculate both metrics, and analyze whether airports with higher RDI have lower average delays.

**Result:** List of airports with RDI and average delay, sorted in ascending order of average delay.

In [ ]:
base_query_4 = base_queries['query_4']

#### **Q5:** How does airline performance differ by type of flight distance (Short, Medium, Long Haul)?

Collection **`us_flights_2023`** is used.
- `distance_type' indicates the flight length category.

For each airline (`airline`) the average delay (`avg(arr_delay)`) is calculated, grouped by `distance_type`.

**Result:** A table with airlines and average delay by route type, sorted descending by average delay within each category.

In [ ]:
base_query_5 = base_queries['query_5']

### **Testing of query execution**

In [30]:
import json

test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Airline" : { "$ne" : None }
        }
    },
    {
        "$lookup" : {
            "from" : "weather_meteo_by_airport",
            "let" : {
                "dep_airport" : "$Dep_Airport",
                "flight_date" : "$FlightDate"
            },
            "pipeline" : [
                {
                    "$match" : {
                        "$expr" : {
                            "$and" : [
                                { "$eq" : [ "$airport_id", "$$dep_airport"] },
                                { "$eq" : [ "$time", "$$flight_date"] },
                                { "$gt" : [ "$prcp", 5.0] },
                                { "$ne" : [ "$prcp", None] }
                            ]
                        }
                    }
                },
                {"$project" : {"_id" : 1, "prcp" : 1}}  # Return min data cuz we just need to know if it exists
            ],
            "as" : "weather_info"
        }
    },
    {
        "$group" : {
            "_id" : "$Airline",
            "average_delay" : { "$avg" : "$Dep_Delay" },
            "flight_count" : { "$sum" : 1 },
            "total_delay_min" : { "$sum" : "$Dep_Delay" }
        }
    },
    {
        "$sort" : { "average_delay" : -1}
    },
    {
        "$limit" : 5
    }
]

try:
    test_results = collection.aggregate(test_pipeline, allowDiskUse=True)
    test_results = list(test_results)
    
    print("Test Results:")
    for i, doc in enumerate(test_results, 1):
        print(f"{i}. Celokupan dokument:")
        print(json.dumps(doc, indent=2, default=str))
        print()
        
except Exception as e:
    print(f"Test Error: {e}")

Test Results:
1. Celokupan dokument:
{
  "_id": "JetBlue Airways",
  "average_delay": 24.27050743706026,
  "flight_count": 267915,
  "total_delay_min": 6502433
}

2. Celokupan dokument:
{
  "_id": "Frontier Airlines Inc.",
  "average_delay": 22.158256417943146,
  "flight_count": 173459,
  "total_delay_min": 3843549
}

3. Celokupan dokument:
{
  "_id": "Spirit Air Lines",
  "average_delay": 19.540724314049715,
  "flight_count": 258838,
  "total_delay_min": 5057882
}

4. Celokupan dokument:
{
  "_id": "American Airlines Inc.",
  "average_delay": 17.43667960407647,
  "flight_count": 928058,
  "total_delay_min": 16182250
}

5. Celokupan dokument:
{
  "_id": "Allegiant Air",
  "average_delay": 15.38555385623771,
  "flight_count": 114425,
  "total_delay_min": 1760492
}



#### **Q3:** Which airports have the most canceled flights during bad weather?

Collections **`cancelled_deverted_2023`**, **`weather_meteo_by_airport`** and **`airports_geolocation`** are used.

Canceled flights are those with `cancelled = 1`.

**Bad weather conditions** are defined as:
* `prcp > 10 mm' *(heavy precipitation)*
* **or** `wspd > 15 m/s' *(strong wind)*

The query should match ($lookup) flights and weather data by `dep_airport` and `airport_id`.

**Result:** Airports with the highest number of canceled flights during bad weather, sorted in descending order of cancellations.

In [33]:
collection = database["cancelled_deverted_2023"]

In [44]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Cancelled" : { "$eq" : 1 }
        }
    },
    {
        "$lookup" : {
            "from" : "weather_meteo_by_airport",
            "let" : {
                "dep_airport" : "$Dep_Airport",
                "flight_date" : "$FlightDate"
            },
            "pipeline" : [
                {
                    "$match" : {
                        "$expr" : {
                            "$and" : [
                                { "$eq" : [ "$airport_id", "$$dep_airport" ]},
                                { "$eq" : [ "$time", "$$flight_date" ]},
                                { "$or" : [
                                    { "$gt" : [ "$prcp", 5.0]},
                                    { "$gt" : [ "$wspd", 15]}
                                ]}
                            ]
                        }
                    }
                },
                {
                    "$project" : { "_id" : 1 }
                }
            ],
            "as" : "weather_info"
        }
    },
    {
        "$match": {
            "weather_info" : {"$ne" : [] }
        }
    },
    {
        "$lookup" : {
            "from" : "airports_geolocation",
            "localField" : "Dep_Airport",
            "foreignField" : "IATA_CODE",
            "as" : "airport_info"
        }
    },

    {
        "$group" : {
            "_id" : "$Dep_Airport",
            "cancelled_flights_count" : { "$sum" : 1 },
            "city" : { "$first" : "$airport_info.CITY" },
            "state" : { "$first" : "$airport_info.STATE" }
        }
    },
    { 
        "$sort" : { "cancelled_flights_count" : -1 }
    }
]

try:
    test_results = collection.aggregate(test_pipeline, allowDiskUse=True)
    test_results = list(test_results)
    
    print("Test Results:")
    for i, doc in enumerate(test_results, 1):
        print(f"{i}. Celokupan dokument:")
        print(json.dumps(doc, indent=2, default=str))
        print()
        
except Exception as e:
    print(f"Test Error: {e}")

Test Results:
1. Celokupan dokument:
{
  "_id": "DFW",
  "cancelled_flights_count": 3870,
  "city": [
    "Dallas-Fort Worth"
  ],
  "state": [
    "TX"
  ]
}

2. Celokupan dokument:
{
  "_id": "DEN",
  "cancelled_flights_count": 2568,
  "city": [
    "Denver"
  ],
  "state": [
    "CO"
  ]
}

3. Celokupan dokument:
{
  "_id": "LGA",
  "cancelled_flights_count": 2504,
  "city": [
    "New York"
  ],
  "state": [
    "NY"
  ]
}

4. Celokupan dokument:
{
  "_id": "JFK",
  "cancelled_flights_count": 2356,
  "city": [
    "New York"
  ],
  "state": [
    "NY"
  ]
}

5. Celokupan dokument:
{
  "_id": "EWR",
  "cancelled_flights_count": 2294,
  "city": [
    "Newark"
  ],
  "state": [
    "NJ"
  ]
}

6. Celokupan dokument:
{
  "_id": "ORD",
  "cancelled_flights_count": 2105,
  "city": [
    "Chicago"
  ],
  "state": [
    "IL"
  ]
}

7. Celokupan dokument:
{
  "_id": "BOS",
  "cancelled_flights_count": 1847,
  "city": [
    "Boston"
  ],
  "state": [
    "MA"
  ]
}

8. Celokupan dokument:
{


Part 2:
```
Test Results:
1. Celokupan dokument:
{
  "_id": "LGA",
  "cancelled_flights_count": 4486
}

2. Celokupan dokument:
{
  "_id": "DFW",
  "cancelled_flights_count": 4414
}

3. Celokupan dokument:
{
  "_id": "DEN",
  "cancelled_flights_count": 4026
}

4. Celokupan dokument:
{
  "_id": "EWR",
  "cancelled_flights_count": 3836
}

5. Celokupan dokument:
{
  "_id": "ORD",
  "cancelled_flights_count": 3152
}

```

In [ ]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Cancelled" : { "$eq" : 1 }
        }
    },
    {
        "$group" : {
            "_id" : "$Dep_Airport",
            "cancelled_flights_count" : { "$sum" : 1 }
        }
    },
    { 
        "$sort" : { "cancelled_flights_count" : -1 }
    },
    {
        "$limit" : 5
    }
]

Part 3:
```
Test Results:
1. Celokupan dokument:
{
  "_id": "LGA",
  "cancelled_flights_count": 4486,
  "city": [
    "New York"
  ],
  "state": [
    "NY"
  ]
}

2. Celokupan dokument:
{
  "_id": "DFW",
  "cancelled_flights_count": 4414,
  "city": [
    "Dallas-Fort Worth"
  ],
  "state": [
    "TX"
  ]
}

3. Celokupan dokument:
{
  "_id": "DEN",
  "cancelled_flights_count": 4026,
  "city": [
    "Denver"
  ],
  "state": [
    "CO"
  ]
}

4. Celokupan dokument:
{
  "_id": "EWR",
  "cancelled_flights_count": 3836,
  "city": [
    "Newark"
  ],
  "state": [
    "NJ"
  ]
}

5. Celokupan dokument:
{
  "_id": "ORD",
  "cancelled_flights_count": 3152,
  "city": [
    "Chicago"
  ],
  "state": [
    "IL"
  ]
}
```

In [ ]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Cancelled" : { "$eq" : 1 }
        }
    },
    {
        "$lookup" : {
            "from" : "airports_geolocation",
            "localField" : "Dep_Airport",
            "foreignField" : "IATA_CODE",
            "as" : "airport_info"
        }
    },
    {
        "$group" : {
            "_id" : "$Dep_Airport",
            "cancelled_flights_count" : { "$sum" : 1 },
            "city" : { "$first" : "$airport_info.CITY" },
            "state" : { "$first" : "$airport_info.STATE" }
        }
    },
    { 
        "$sort" : { "cancelled_flights_count" : -1 }
    },
    {
        "$limit" : 5
    }
]

```
Test Results:
1. Celokupan dokument:
{
  "_id": "DFW",
  "cancelled_flights_count": 3870,
  "city": [
    "Dallas-Fort Worth"
  ],
  "state": [
    "TX"
  ]
}

2. Celokupan dokument:
{
  "_id": "DEN",
  "cancelled_flights_count": 2568,
  "city": [
    "Denver"
  ],
  "state": [
    "CO"
  ]
}

3. Celokupan dokument:
{
  "_id": "LGA",
  "cancelled_flights_count": 2504,
  "city": [
    "New York"
  ],
  "state": [
    "NY"
  ]
}

4. Celokupan dokument:
{
  "_id": "JFK",
  "cancelled_flights_count": 2356,
  "city": [
    "New York"
  ],
  "state": [
    "NY"
  ]
}

5. Celokupan dokument:
{
  "_id": "EWR",
  "cancelled_flights_count": 2294,
  "city": [
    "Newark"
  ],
  "state": [
    "NJ"
  ]
}
```

In [ ]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Cancelled" : { "$eq" : 1 }
        }
    },
    {
        "$lookup" : {
            "from" : "weather_meteo_by_airport",
            "let" : {
                "dep_airport" : "$Dep_Airport",
                "flight_date" : "$FlightDate"
            },
            "pipeline" : [
                {
                    "$match" : {
                        "$expr" : {
                            "$and" : [
                                { "$eq" : [ "$airport_id", "$$dep_airport" ]},
                                { "$eq" : [ "$time", "$$flight_date" ]},
                                { "$or" : [
                                    { "$gt" : [ "$prcp", 5.0]},
                                    { "$gt" : [ "$wspd", 15]}
                                ]}
                            ]
                        }
                    }
                },
                {
                    "$project" : { "_id" : 1 }
                }
            ],
            "as" : "weather_info"
        }
    },
    {
        "$match": {
            "weather_info" : {"$ne" : [] }
        }
    },
    {
        "$lookup" : {
            "from" : "airports_geolocation",
            "localField" : "Dep_Airport",
            "foreignField" : "IATA_CODE",
            "as" : "airport_info"
        }
    },

    {
        "$group" : {
            "_id" : "$Dep_Airport",
            "cancelled_flights_count" : { "$sum" : 1 },
            "city" : { "$first" : "$airport_info.CITY" },
            "state" : { "$first" : "$airport_info.STATE" }
        }
    },
    { 
        "$sort" : { "cancelled_flights_count" : -1 }
    }
]


**Q4:** Do airports with more diverse runways have lower average delays?

Collections **`airports`**, **`runways`**, and **`us_flights_2023`** are used.

For each airport, the following is calculated:
* **Runway Diversity Index (RDI)** = number of different values ​​of `surface` from the collection of `runways` per airport.
* **Average Delay** = average `dep_delay` from `us_flights_2023` per airport.

It is necessary to merge (`$lookup') all three collections, calculate both metrics, and analyze whether airports with higher RDI have lower average delays.

**Result:** List of airports with RDI and average delay, sorted in ascending order of average delay.

In [49]:
collection = database["airports"]

In [55]:
test_pipeline = [
    {
        "$match" : {
            "iata_code" : { "$ne" : None },
            "type" : { "$in" : ["medium_airport", "large_airport"]}
        }
    },
    {
        "$lookup" : {
            "from" : "runways",
            "localField" : "ident",
            "foreignField" : "airport_ident",
            "as" : "runway_info"
        },
    },
    {
        "$lookup" : {
            "from" : "us_flights_2023",
            "let" : {
                "airport_code" : "$iata_code"
            },
            "pipeline" : [
                {
                    "$match" : {
                        "$expr" : {
                            "$eq" : [ "$Dep_Airport", "$$airport_code" ]
                        }
                    }
                },
                {
                    "$group" : {
                        "_id" : "$Dep_Airport",
                        "average_delay" : { "$avg" : "$Dep_Delay" },
                        "total_flights" : { "$sum" : 1 }
                    }
                }
            ],
            "as" : "flight_info"
        }
    },
    {
        "$match" : {
            "runway_info" : { "$ne" : [] },
            "flight_info" : { "$ne" : [] }
        }
    },
    {
        "$sort" : { "average_delay" : -1, "total_flights" : -1 }
    }
]

try:
    test_results = collection.aggregate(test_pipeline, allowDiskUse = True)
    test_results = list(test_results)

    print("Test Results:")
    for i, doc in enumerate(test_results, 1):
        print(f"{i}. Ceo dokument:")
        print(json.dumps(doc, indent=2, default=str))
        print()
except Exception as e:
    print(f"Test Error: {e}")

Test Results:
1. Ceo dokument:
{
  "_id": "68f2e8a7bb35bab1fe21024c",
  "id": 3356,
  "ident": "KABE",
  "type": "medium_airport",
  "name": "Lehigh Valley International Airport",
  "latitude_deg": 40.652099609375,
  "longitude_deg": -75.44080352783203,
  "elevation_ft": 393.0,
  "iso_country": "US",
  "iso_region": "US-PA",
  "municipality": "Allentown",
  "scheduled_service": "yes",
  "gps_code": "KABE",
  "iata_code": "ABE",
  "local_code": "ABE",
  "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport",
  "runway_info": [
    {
      "_id": "68f2e89bbb35bab1fe2039b7",
      "id": 240632,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "length_ft": 7600,
      "width_ft": 150,
      "surface": "ASP",
      "lighted": 1,
      "closed": 0,
      "le_ident": 6,
      "le_latitude_deg": 40.647,
      "le_longitude_deg": -75.4506,
      "le_elevation_ft": 394,
      "le_heading_degT": 51.3,
      "he_ident": 24,
      "he_latitude_deg": 40.